# GeoSR-AI — Notebook 03: SRCNN Neural Network Baseline
**Deep Learning Based Super Resolution Mapping from Medium-Resolution Satellite Imagery**

### Overview
This notebook covers:
1. **SRCNN Architecture** (Super-Resolution Convolutional Neural Network) implementation
2. L1 Reconstruction Loss optimization
3. PyTorch training & validation loop with checkpointing (`best_model.pth`, `last_model.pth`)
4. Quantitative evaluation and comparison against Bicubic baseline

In [1]:
import os
import sys
import torch
import yaml
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from models.srcnn import SRCNN
from datasets.paired_dataset import create_dataloaders
from training.trainer import GeoSRTrainer
from evaluation.benchmark import evaluate_model


## 1. Initialize SRCNN Model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
srcnn_model = SRCNN(in_channels=3, out_channels=3, num_features=64, scale_factor=4).to(device)
print(srcnn_model)
print(f"Total trainable parameters: {sum(p.numel() for p in srcnn_model.parameters() if p.requires_grad):,}")

SRCNN(
  (conv1): Conv2d(3, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4))
  (conv2): Conv2d(64, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv3): Conv2d(32, 3, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu): ReLU(inplace=True)
)
Total trainable parameters: 69,251


## 2. Train SRCNN Model

In [3]:
with open("../configs/srcnn.yaml", "r") as f:
    config = yaml.safe_load(f)

train_loader, val_loader, test_loader = create_dataloaders(
    data_dir="../data/processed",
    batch_size=config['training']['batch_size'],
    patch_size=config['data']['patch_size'],
    scale_factor=config['data']['scale_factor'],
    num_workers=0
)

trainer = GeoSRTrainer(model=srcnn_model, config=config, device=device)
history = trainer.fit(train_loader=train_loader, val_loader=val_loader, epochs=5)
print("Training completed successfully.")


Starting GeoSR Model Training on cpu for 5 epochs...
Epoch [01/05] | Train Loss: 0.0356 | Val Loss: 0.0533 | Val PSNR: 22.92 dB | Val SSIM: 0.9744 | Time: 717.4s
Epoch [02/05] | Train Loss: 0.0314 | Val Loss: 0.0564 | Val PSNR: 22.29 dB | Val SSIM: 0.9725 | Time: 818.0s
Epoch [03/05] | Train Loss: 0.0312 | Val Loss: 0.0561 | Val PSNR: 22.32 dB | Val SSIM: 0.9723 | Time: 692.8s
Epoch [04/05] | Train Loss: 0.0311 | Val Loss: 0.0549 | Val PSNR: 22.44 dB | Val SSIM: 0.9724 | Time: 692.7s
Epoch [05/05] | Train Loss: 0.0310 | Val Loss: 0.0548 | Val PSNR: 22.60 dB | Val SSIM: 0.9719 | Time: 731.5s
Training completed successfully.
